[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jalonso1979/puremacro/blob/main/curso/notebooks/T03_E_local_llm_uncertainty_es.ipynb)

> **Nota para Google Colab**: la celda siguiente instala `puremacro` y, si hace falta, descarga del repositorio público [jalonso1979/puremacro](https://github.com/jalonso1979/puremacro) sólo la carpeta `curso/notebooks` (auxiliares, datos congelados y modelos). No pide acceso a Google Drive.

In [1]:
# Configuración del entorno: Google Colab, instalación local o Juno (iPad)
import os
import sys
from pathlib import Path

def _safe_probe(p, is_dir=None):
    """¿Existe la ruta? Sin lanzar OSError/PermissionError en el sandbox de Juno."""
    try:
        p = Path(p)
        if is_dir is True:
            return p.is_dir()
        if is_dir is False:
            return p.is_file()
        return p.exists()
    except (OSError, PermissionError, TypeError, ValueError):
        return False

if "google.colab" in sys.modules:
    !pip install -q "puremacro>=4.8.1,<5"
    if not _safe_probe(Path.cwd() / "_nbstyle.py", is_dir=False):
        # Clon parcial del repositorio público: sólo curso/notebooks, sin historia
        _repo = Path("/content/puremacro")
        if not _safe_probe(_repo, is_dir=True):
            !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/jalonso1979/puremacro.git {_repo}
        !git -C {_repo} sparse-checkout set curso/notebooks 2>&1 | tail -n 2
        os.chdir(_repo / "curso/notebooks")

# Carpeta con los auxiliares del curso (_nbstyle, _datos) en sys.path, sin salir del sandbox
_candidatos = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd() / "curso/notebooks"]
try:
    _candidatos.insert(0, Path(__file__).resolve().parent)  # Juno define __file__
except (NameError, OSError):
    pass
for _c in _candidatos:
    if _safe_probe(_c / "_nbstyle.py", is_dir=False):
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

# Gráficas en línea y estilo del curso
%matplotlib inline
try:
    import _nbstyle
    _nbstyle.apply_style()
except ImportError:
    pass

# Modelos de Lenguaje como Herramientas de Medición en Macroeconomía

La investigación macroeconómica moderna requiere con frecuencia transformar texto narrativo no estructurado en indicadores cuantitativos: minutas de política monetaria, proyectos legislativos, reportes de supervisión financiera y transcripciones de bancos centrales.

Históricamente, la literatura de texto-como-datos utilizó diccionarios de términos y conteos booleanos (por ejemplo, el índice EPU de Baker-Bloom-Davis o el léxico de Loughran y McDonald 2011). Aunque computacionalmente eficientes y deterministas, los métodos de diccionario presentan limitaciones econométricas insoslayables:
1. **Negación e Inversión Contextual**: Una frase como *"El comité no prevé un incremento cercano en las presiones inflacionarias"* activa un conteo positivo en un diccionario de inflación, distorsionando el sentido macroeconómico real.
2. **Polisemia y Ambigüedad Sintáctica**: La palabra *"tasa"* alude a tasas de interés, tasas de desempleo o depreciación cambiaria según el contexto de la oración.
3. **Inflexibilidad Temporal y Estilística**: Los listados estáticos de palabras no captan giros discursivos emergentes (tales como *"endurecimiento cuantitativo"*, *"postura neutral"* o *"anclaje de expectativas"*).

Los Modelos de Lenguaje de Gran Escala (LLMs) superan estas restricciones mediante mecanismos de auto-atención profunda (Vaswani et al. 2017), transformando los textos en representaciones semánticas continuas capaces de interpretar sintaxis, subordinación y matices condicionantes.

No obstante, la incorporación de LLMs en la macroeconomía empírica introduce desafíos institucionales y metodológicos de primer orden:
- **Privacidad y Confidencialidad Institucional**: Transmitir minutas de política monetaria sujetas a embargo, información bancaria reservada o datos corporativos a APIs comerciales en la nube vulnera mandatos regulatorios.
- **Replicabilidad Científica**: Las APIs en la nube actualizan sus pesos y alineaciones sin aviso previo, comprometiendo la reproducibilidad de los resultados econométricos a lo largo del tiempo.
- **Estocasticidad y Alucinación**: Salvo que se elimine la aleatoriedad (fijando la temperatura en cero) y se calibren las probabilidades de clasificación, el modelo introduce errores de medición que sesgan las estimaciones en modelos de series de tiempo.

Este cuaderno examina el despliegue econométrico de **modelos de lenguaje locales y autónomos** (Apple MLX, llama.cpp, formatos cuantizados GGUF). Comparamos la extracción por diccionario frente a la clasificación semántica en choques de política fiscal, evaluamos kernels de probabilidad de incertidumbre y establecemos estándares de reproducibilidad para la macroeconomía empírica.

In [2]:
import _llm  # fija la ruta de la caché del curso
import sys
from pathlib import Path

_cwd = Path.cwd()
sys.path.insert(0, str(_cwd if _safe_probe(_cwd / "_nbstyle.py", is_dir=False) else _cwd / "notebooks"))
import _nbstyle
_nbstyle.apply_style()

from puremacro.narrative.scoring import (
    score_llm,
    score_keyword,
)

# Corpus macroeconómico de referencia: intervenciones de política, despachos de incertidumbre y reportes neutrales
CORPUS = [
    (
        "2020-03-15",
        "The federal government enacted a $500 billion emergency fiscal stimulus and infrastructure package to support domestic liquidity.",
        "https://macro-corpus.example/doc_001",
    ),
    (
        "2020-04-01",
        "Central bank officials warned that the macroeconomic outlook is highly uncertain and could deteriorate abruptly if financial stress persists.",
        "https://macro-corpus.example/doc_002",
    ),
    (
        "2021-06-10",
        "The trade ministry reported that the bilateral trade surplus expanded moderately during the second quarter as port operations normalized.",
        "https://macro-corpus.example/doc_003",
    ),
    (
        "2022-09-22",
        "The committee announced a 75 basis point rate increase, noting that monetary tightening would continue until price stability is restored.",
        "https://macro-corpus.example/doc_004",
    ),
]

print(f"Corpus inicializado con {len(CORPUS)} documentos macroeconómicos representativos.")

Corpus inicializado con 4 documentos macroeconómicos representativos.


## 1. Arquitectura Local Offline: Garantizando Reproducibilidad y Seguridad de Datos

En el análisis de política macroeconómica y banca central, la analítica textual debe cumplir dos requisitos operativos esenciales:
1. **Despliegue Local Aislado**: Los modelos deben ejecutarse directamente en la estación de trabajo del investigador o en servidores institucionales seguros (mediante Apple MLX en Apple Silicon, llama.cpp o modelos cuantizados como Qwen3-4B, el modelo del curso, o Llama-3.2-3B). Ningún token sale a redes externas, garantizando el cumplimiento de normas de confidencialidad de bancos centrales e institutos de estadística.
2. **Respuestas Reproducibles Sin el Modelo**: Cada respuesta se guarda bajo una clave formada por el modelo y el prompt exacto, así que una corrida puede repetirse, y revisarse, sin el modelo. Esta lección llama al modelo a través del ayudante del curso `_llm` (lección T00_B), que lee la caché del curso que viaja con los cuadernos, `data_curso/llm_cache.sqlite`: las respuestas del modelo de referencia, Qwen3-4B, calculadas una vez por el profesor. Por eso las celdas de abajo muestran salidas reales del modelo sin ningún modelo instalado; con un modelo local instalado, lo que la caché no tiene se responde en vivo y se guarda. Desde puremacro 5.0, las llamadas de la propia librería (`score_llm` con un backend local, `llm_prob_kernel` con un proveedor local) leen la misma caché. Los simulados deterministas de `puremacro` (`MockBackend`, `MockProvider`) siguen para las pruebas de software que no deben depender de ningún modelo.

In [3]:
# El modelo del curso, a través del ayudante `_llm`: score_llm recibe llm.backend() y el índice
# de incertidumbre sale de llm.index(...). Cada respuesta se busca primero en la caché del curso;
# en modo cache no corre nada más, y en modo auto un modelo local responde lo que falte.
llm = _llm.get_llm()

print(f"Motor de inferencia activo:  {llm.backend()!r}")
print(f"Proveedor narrativo activo:  {llm.provider()!r}")

Motor de inferencia activo:  CourseBackend(model='qwen3-4b', mode='cache')
Proveedor narrativo activo:  CourseProvider(model='qwen3-4b', mode='cache')


## 2. Extracción de Información: Diccionarios vs Análisis Semántico en Choques Fiscales

Un desafío clásico en macroeconomía empírica (Romer y Romer 2010; Cloyne 2013) consiste en aislar choques fiscales discrecionales exógenos de los estabilizadores automáticos endógenos. Para ello, el investigador debe extraer tres variables clave a partir del texto legislativo:
1. **Fecha de la Medida**: Momento en que la ley fue aprobada o anunciada.
2. **Dirección (Signo)**: Medida expansiva ($+1$) frente a restrictiva ($-1$).
3. **Magnitud y Unidades**: Tamaño del impulso fiscal en miles de millones de moneda o puntos del PIB.

A continuación comparamos:
- **Puntuación por Diccionario y Expresiones Regulares** (`score_keyword`): Aplica listados léxicos predefinidos y expresiones regulares numéricas.
- **Extracción Semántica con LLM** (`score_llm`): Emplea prompting estructurado zero-shot para inferir fechas, signo, destino del gasto (infraestructura vs transferencias) y magnitudes cuantitativas en estructuras `NarrativeEvent`.

Los eventos del método semántico salen de las respuestas de Qwen3-4B en la caché del curso: el modelo leyó cada documento una vez, cuando el profesor construyó la caché, y la celda repite su JSON. Un documento cuya respuesta falta en la caché imprime una nota `[modelo local]` y no aporta eventos; la lección sigue.

In [4]:
# 1. Extracción por diccionario (coincidencia de palabras clave según reglas)
events_kw = score_keyword(CORPUS, kind="fiscal")
print(f"Método de diccionario: {len(events_kw)} evento(s) fiscal(es) extraído(s)")
for ev in events_kw:
    print(f"  [{ev.date.date()}] Destino: {ev.target}/{ev.subtarget} | Signo: {ev.sign:+d} | Magnitud: {ev.magnitude} {ev.magnitude_unit}")

# 2. Extracción semántica con LLM (lectura estructurada zero-shot)
events_llm = score_llm(CORPUS, backend=llm.backend(), kind="fiscal")
print(f"Método semántico (LLM): {len(events_llm)} evento(s) fiscal(es) extraído(s)")
for ev in events_llm:
    print(f"  [{ev.date.date()}] Signo: {ev.sign:+d} | Magnitud: {ev.magnitude} {ev.magnitude_unit}")

# Verificación econométrica:
# El diccionario de referencia detecta con fiabilidad el choque explícito de inversión en infraestructura del Documento 0
assert len(events_kw) >= 1, "Dictionary method should identify explicit fiscal events"
assert events_kw[0].sign == 1, "Fiscal stimulus must be flagged as expansionary (sign=+1)"

Método de diccionario: 1 evento(s) fiscal(es) extraído(s)
  [2020-03-15] Destino: investment/infra | Signo: +1 | Magnitud: 500.0 USD_bn
Método semántico (LLM): 1 evento(s) fiscal(es) extraído(s)
  [2020-03-15] Signo: +1 | Magnitud: 500.0 USD_bn


## 3. Incertidumbre Narrativa: Kernels de Probabilidad y Sensibilidad de Prompts

Más allá de la extracción de eventos discretos, la macroeconomía empírica requiere frecuentemente estimar probabilidades continuas a partir del texto:
$$ P(\text{Incertidumbre} \mid d_t) \in [0, 1] $$
que expresa la probabilidad condicional de que el documento $d_t$ transmita ambigüedad o volatilidad de política.

`llm.index(CORPUS, categoria)` corre `llm_prob_kernel()` de puremacro con el modelo del curso: el núcleo divide cada documento en ventanas (aquí un párrafo, el documento entero), le pregunta al modelo con qué fuerza expresa cada ventana la categoría en una escala de 0.0 a 1.0 y promedia las respuestas de cada documento.

### Sensibilidad de Prompts y Calibración de Temperatura
En aplicaciones econométricas, los modelos de lenguaje deben configurarse bajo protocolos rigurosos:
1. **Decodificación Voraz (Greedy, $\tau = 0$)**: Una temperatura superior a cero introduce varianza estocástica en el muestreo de tokens, impidiendo la replicabilidad exacta entre ejecuciones. Fijar $\tau = 0$ garantiza determinismo matemático.
2. **Definición de Fronteras de Categoría**: Las instrucciones deben explicitar criterios de exclusión (p. ej., distinguir entre incertidumbre genuina de política macroeconómica y cobertura rutinaria del ciclo de negocios) para evitar derivas semánticas.

In [5]:
# Calcula puntajes semánticos continuos de incertidumbre en todo el corpus
prob_series = llm.index(CORPUS, "economic uncertainty")

print("Probabilidades continuas de incertidumbre narrativa:")
for date, p in prob_series:
    print(f"  [{date.date()}] P(Incertidumbre económica | Documento) = {p:.3f}")

# Aserciones de verificación
probs_scored = [p for _, p in prob_series]

assert len(prob_series) == len(CORPUS), "Every document in the corpus must receive a score"
assert all(0.0 <= p <= 1.0 for p in probs_scored), "Probabilities must be strictly bounded in [0, 1]"

Probabilidades continuas de incertidumbre narrativa:
  [2020-03-15] P(Incertidumbre económica | Documento) = 0.300
  [2020-04-01] P(Incertidumbre económica | Documento) = 0.800
  [2021-06-10] P(Incertidumbre económica | Documento) = 0.200
  [2022-09-22] P(Incertidumbre económica | Documento) = 0.300


## 4. Entre el diccionario y el LLM: la matriz documento-término

Entre una lista de palabras y un modelo de lenguaje está la representación de la que parte casi todo el trabajo con texto como datos: la matriz documento-término (DTM, por sus siglas en inglés). Tiene una fila por documento y una columna por palabra del vocabulario, y cada celda cuenta cuántas veces aparece la palabra en el documento. TF-IDF repondera cada columna según lo rara que es la palabra entre documentos, $\text{idf}_j = \ln\frac{1+n}{1+\text{df}_j} + 1$, donde $n$ es el número de documentos y $\text{df}_j$ el número de los que contienen la palabra $j$, y después escala cada fila a longitud uno. Los índices tipo EPU, los modelos de tópicos y la mayoría de las regresiones sobre texto toman esta matriz como insumo; T03_D escribe el índice de Baker-Bloom-Davis como álgebra sobre ella.

La celda siguiente construye la DTM de los mismos cuatro documentos con `puremacro.text.build_dtm`, tras quitar las palabras vacías del inglés, y reporta el peso TF-IDF de *uncertain* en cada uno. `puremacro.text` aún no está en la versión publicada de `puremacro`; si la importación falla, esa celda lo avisa y el resto del cuaderno corre.

In [6]:
# Matriz documento-término del mismo corpus (necesita puremacro.text)
try:
    from puremacro.text import build_dtm
    HAY_TEXT = True
except ImportError:
    HAY_TEXT = False
    print("Esta sección necesita puremacro 5.0 (aún no publicado); en Colab instala la versión de desarrollo con\n"
          "    pip install git+https://github.com/jalonso1979/puremacro\n"
          "o salta a la sección siguiente.")

if HAY_TEXT:
    import numpy as np

    dtm = build_dtm([text for _, text, _ in CORPUS], languages="en")
    tfidf = dtm.tfidf()
    print(f"Matriz documento-término: {dtm.n_docs} documentos x {dtm.n_terms} términos")
    print(f"Términos que aparecen en más de un documento: {int((dtm.document_frequency() > 1).sum())}")

    j = dtm.term_index["uncertain"]
    w_unc = tfidf.X[:, j].toarray().ravel()
    print("Peso TF-IDF de 'uncertain':")
    for (date, _, _), w in zip(CORPUS, w_unc):
        print(f"  [{date}] {w:.3f}")

    row = tfidf.X[1].toarray().ravel()
    n_tied = int(np.isclose(row, w_unc[1]).sum())
    print(f"Términos del documento del 2020-04-01 con ese mismo peso: {n_tied} de {int((row > 0).sum())}")

    assert dtm.n_docs == len(CORPUS)
    assert w_unc[1] > 0 and np.all(np.delete(w_unc, 1) == 0)
    same = [w > 0 for w in w_unc] == [p >= 0.5 for p in probs_scored]
    print(f"Marca los mismos documentos que el modelo del curso (P >= 0.5): {same}")

Matriz documento-término: 4 documentos x 52 términos
Términos que aparecen en más de un documento: 0
Peso TF-IDF de 'uncertain':
  [2020-03-15] 0.000
  [2020-04-01] 0.277
  [2021-06-10] 0.000
  [2022-09-22] 0.000
Términos del documento del 2020-04-01 con ese mismo peso: 13 de 13
Marca los mismos documentos que el modelo del curso (P >= 0.5): True


La matriz tiene 4 filas y 52 columnas, y ningún término aparece en más de un documento: sin las palabras vacías, los cuatro textos no comparten vocabulario. Como todas las frecuencias documentales valen 1, el idf es el mismo para todas las columnas y TF-IDF se reduce a conteos escalados por fila. *Uncertain* pesa 0,277 en el comunicado de abril de 2020 y 0 en los otros tres, así que la matriz marca un documento. Dentro de ese comunicado, sin embargo, los 13 términos tienen el mismo peso de 0,277: *uncertain* cuenta lo mismo que *officials* o *bank*. La matriz registra qué palabras aparecen dónde, no lo que dicen. Para convertirla en una medida de incertidumbre hay que escoger las columnas, y eso es volver a un diccionario; un modelo de lenguaje debería hacer esa lectura a partir del contexto. Las puntuaciones del modelo del curso en la sección 3 son una lectura así de estas cuatro frases; si lee mejor que un diccionario sólo puede juzgarse con un corpus de más de cuatro frases.

## Directrices Metodológicas para la Medición con LLMs en Macroeconomía

Para salvaguardar el rigor científico al emplear Modelos de Lenguaje como instrumentos de medición econométrica, los investigadores deben observar cuatro principios metodológicos:

1. **Ejecución Determinista**: Fijar siempre la temperatura en cero ($\text{temperatura} = 0$) y establecer semillas deterministas para asegurar la replicabilidad exacta de las series construidas.
2. **Despliegue Local en Hardware Institucional**: Ejecutar modelos abiertos y cuantizados (MLX, GGUF mediante llama.cpp) en servidores locales para preservar la confidencialidad de la información de bancos centrales y eliminar la dependencia de servicios comerciales externos.
3. **Validación Cruzada con Diccionarios y Auditoría Humana**: Contrastar las clasificaciones automáticas con índices canónicos de diccionario (como Baker-Bloom-Davis EPU) y muestras etiquetadas manualmente a doble ciego. Reportar matrices de confusión, precisión, sensibilidad y coeficientes de concordancia (kappa de Cohen).
4. **Tratamiento del Error de Medición en Modelos Posteriores**: Tratar las variables narrativas como regresores sujetos a error de medición ($x_t^* = x_t + \eta_t$). Emplear variables instrumentales o correcciones de errores en las variables (Fuller 1987) para evitar el sesgo de atenuación al estimar multiplicadores dinámicos en modelos VAR o proyecciones locales.